# 01. 세탁 거래만 분리 — 유형분류 데이터 준비

`01_prepare_parquet` 출력(`transactions.parquet`, `patterns.parquet`)에서 **Is Laundering=1 거래만** 꺼내 유형 라벨을 붙인다.
유형 모델 입력은 "의심 거래만 모은 그래프"로 정했으므로, 첫 실험(상한선)은 세탁 거래만으로 그래프·라벨을 만든다.

**라벨 규칙 (9클래스)**
- 패턴 파일에 있는 거래 → 그 시도의 `pattern_type` (8개)
- 패턴 파일에 없는 세탁 거래 → `UNMATCHED` (무패턴 세탁)
- 한 거래가 여러 시도에 들어 있으면 **번호가 가장 작은 시도**의 유형을 쓰고, `n_attempts`·`types` 로 표시만 해 둔다 (2절에서 몇 건인지 확인)

**출력** `parquets/<DATASET>/multiclass/`
| 파일 | 내용 |
|---|---|
| `laundering.parquet` | 세탁 거래 1행 = 1엣지. 거래 열 + `pattern_type`, `class_id`, `attempt_id`, `n_attempts`, `types`, `in_pattern` |
| `attempts.parquet` | 시도 1행. `pattern_type`, `n_rows`(파일 행), `n_edges`, `n_edges_labeled`(이 시도 유형으로 라벨된 엣지), `n_accounts`, `start_ts`, `end_ts`, `duration_h` |
| `daily_all.parquet` | 일별 전체 거래 수·세탁 거래 수 (정상 끊기는 날 확인·그림용) |
| `summary.json` | 건수 요약 |

전 기간(11월 꼬리 포함)을 저장한다. 어느 기간을 쓸지는 02 분석에서 정한다.

## 0. 설정

In [1]:
import sys, json, time
from pathlib import Path
import duckdb, pandas as pd

sys.path.append(str(Path.cwd()))
import mc_lib as M

DATASET = "HI-Large"                 # HI-Small 로 먼저 돌려 봐도 됨
THREADS, MEMORY = 8, "12GB"

BASE = M.base_dir(DATASET)
TX, PAT = BASE / "transactions.parquet", BASE / "patterns.parquet"
OUT = M.mc_dir(DATASET)
assert TX.is_file() and PAT.is_file(), f"01_prepare_parquet 을 {DATASET} 으로 먼저 실행: {BASE}"

tmp = OUT / "duckdb_tmp"; tmp.mkdir(exist_ok=True)
con = duckdb.connect()
con.execute(f"SET threads={THREADS}"); con.execute(f"SET memory_limit='{MEMORY}'")
con.execute(f"SET temp_directory={M.q(tmp)}"); con.execute("SET preserve_insertion_order=false")
q = M.q
summary = {"dataset": DATASET}
print(BASE, "→", OUT)

/workspace/money_laundry_ai_/parquets/HI-Large → /workspace/money_laundry_ai_/parquets/HI-Large/multiclass


## 1. 세탁 거래 추출 + 일별 전체 거래 수

`transactions.parquet` 를 두 번 스캔한다 (세탁 행 필터, 일별 집계). Large 기준 각 1분 안쪽.
중복 대표만 들어 있는 파일이라(01 규칙) 중복 처리는 다시 하지 않는다.

In [2]:
t0 = time.perf_counter()
con.execute(f"""
    CREATE OR REPLACE TABLE lt AS
    SELECT edge_id, ts, day, src, dst, from_bank, to_bank, paid, received,
           payment_currency, receiving_currency, payment_format
    FROM read_parquet({q(TX)}) WHERE is_laundering""")
con.execute(f"""
    CREATE OR REPLACE TABLE daily_all AS
    SELECT day, count(*) AS n_all, count(*) FILTER (WHERE is_laundering) AS n_laundering
    FROM read_parquet({q(TX)}) GROUP BY day ORDER BY day""")
summary["laundering_edges"] = con.execute("SELECT count(*) FROM lt").fetchone()[0]
print(f"세탁 거래 {summary['laundering_edges']:,}  ({time.perf_counter()-t0:.0f}s)")

daily = con.execute("SELECT * FROM daily_all ORDER BY day").df()
daily["n_normal"] = daily["n_all"] - daily["n_laundering"]
summary["last_day_all"] = str(daily["day"].max())
summary["last_day_normal"] = str(daily.loc[daily["n_normal"] > 0, "day"].max())
summary["last_day_laundering"] = str(daily.loc[daily["n_laundering"] > 0, "day"].max())
print("마지막 날 — 전체:", summary["last_day_all"], " 정상:", summary["last_day_normal"], " 세탁:", summary["last_day_laundering"])
daily.tail(12)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

세탁 거래 225,546  (10s)
마지막 날 — 전체: 2023-01-12 00:00:00  정상: 2023-01-12 00:00:00  세탁: 2023-01-12 00:00:00


,day,n_all,n_laundering,n_normal
153,2023-01-01,45,26,19
154,2023-01-02,26,17,9
155,2023-01-03,38,24,14
156,2023-01-04,26,17,9
157,2023-01-05,30,17,13
158,2023-01-06,31,19,12
159,2023-01-07,11,6,5
160,2023-01-08,13,9,4
161,2023-01-09,13,8,5
162,2023-01-10,8,5,3


## 2. 패턴 소속 확인

`patterns.parquet` 는 패턴 파일 1행 = 1행 (시도 × 거래). 확인할 것
- 거래와 매칭 안 된 패턴 행 (`unmatched`)
- 한 거래가 여러 시도에 들어 있는지 (`n_attempts > 1`), 유형까지 다른지 (`n_types > 1`)
- 같은 시도 안에 같은 거래가 두 번 (`n_rows > n_attempts`) — 원본 중복 행이 대표 1건으로 합쳐진 경우
- 패턴 거래인데 라벨이 0 인 것 (있으면 안 됨)

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE pm AS
    SELECT attempt_id, pattern_type, pattern_desc, edge_id, ts, unmatched FROM read_parquet({q(PAT)})""")
con.execute("""
    CREATE OR REPLACE TABLE em AS
    SELECT edge_id,
           min(attempt_id)                       AS attempt_id,     -- 라벨에 쓸 시도 = 번호가 가장 작은 시도
           arg_min(pattern_type, attempt_id)     AS pattern_type,
           count(*)                              AS n_rows,
           count(DISTINCT attempt_id)            AS n_attempts,
           count(DISTINCT pattern_type)          AS n_types,
           string_agg(DISTINCT pattern_type, '|' ORDER BY pattern_type) AS types
    FROM pm WHERE edge_id IS NOT NULL GROUP BY edge_id""")

chk = con.execute("""
    SELECT (SELECT count(*) FROM pm)                                   AS pattern_rows,
           (SELECT count(*) FROM pm WHERE unmatched)                   AS pattern_rows_unmatched,
           (SELECT count(DISTINCT attempt_id) FROM pm)                 AS attempts,
           (SELECT count(*) FROM em)                                   AS pattern_edges,
           (SELECT count(*) FROM em WHERE n_attempts > 1)              AS edges_multi_attempt,
           (SELECT count(*) FROM em WHERE n_types > 1)                 AS edges_multi_type,
           (SELECT count(*) FROM em WHERE n_rows > n_attempts)         AS edges_dup_in_attempt,
           (SELECT count(*) FROM em LEFT JOIN lt USING (edge_id) WHERE lt.edge_id IS NULL) AS pattern_edges_label0
""").df()
summary.update({k: int(v) for k, v in chk.iloc[0].items()})
chk.T.rename(columns={0: "count"})

,count
pattern_rows,137936
pattern_rows_unmatched,0
attempts,16467
pattern_edges,137936
edges_multi_attempt,0
edges_multi_type,0
edges_dup_in_attempt,0
pattern_edges_label0,0


In [4]:
# 여러 시도에 들어간 거래 — 어떤 유형 조합인가
con.execute("""SELECT types, n_attempts, count(*) AS edges FROM em WHERE n_attempts > 1
               GROUP BY ALL ORDER BY edges DESC LIMIT 20""").df()

,types,n_attempts,edges


In [5]:
# 매칭 안 된 패턴 행 — 유형별
con.execute("""SELECT pattern_type, count(*) AS rows, count(*) FILTER (WHERE unmatched) AS unmatched
               FROM pm GROUP BY 1 ORDER BY 1""").df()

,pattern_type,rows,unmatched
0,BIPARTITE,12174,0
1,CYCLE,12340,0
2,FAN-IN,13287,0
3,FAN-OUT,13554,0
4,GATHER-SCATTER,26361,0
5,RANDOM,9618,0
6,SCATTER-GATHER,26978,0
7,STACK,23624,0


## 3. 라벨 붙이기 → `laundering.parquet`

In [6]:
con.register("classes", pd.DataFrame({"pattern_type": M.CLASSES, "class_id": range(len(M.CLASSES))}))
con.execute(f"""
    COPY (
        SELECT l.*,
               coalesce(e.pattern_type, '{M.UNMATCHED}') AS pattern_type,
               c.class_id,
               e.attempt_id,
               coalesce(e.n_attempts, 0)  AS n_attempts,
               e.types,
               e.edge_id IS NOT NULL      AS in_pattern
        FROM lt l
        LEFT JOIN em e USING (edge_id)
        LEFT JOIN classes c ON c.pattern_type = coalesce(e.pattern_type, '{M.UNMATCHED}')
        ORDER BY l.ts, l.edge_id
    ) TO {q(OUT / 'laundering.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)""")

lab = con.execute(f"""SELECT pattern_type, class_id, count(*) AS edges, count(DISTINCT attempt_id) AS attempts,
                              min(ts) AS first_ts, max(ts) AS last_ts
                       FROM read_parquet({q(OUT / 'laundering.parquet')}) GROUP BY 1, 2 ORDER BY class_id""").df()
assert lab["class_id"].notna().all(), f"CLASSES 에 없는 유형: {lab.loc[lab.class_id.isna(), 'pattern_type'].tolist()}"
summary["class_edges"] = dict(zip(lab["pattern_type"], lab["edges"].astype(int)))
lab

,pattern_type,class_id,edges,attempts,first_ts,last_ts
0,FAN-OUT,0,13554,2040,2022-08-01 00:36:00,2022-12-10 15:28:00
1,FAN-IN,1,13287,2014,2022-08-01 00:08:00,2022-12-10 00:40:00
2,GATHER-SCATTER,2,26361,2054,2022-08-01 00:10:00,2023-01-12 16:49:00
3,SCATTER-GATHER,3,26978,2067,2022-08-01 02:10:00,2022-12-10 18:19:00
4,CYCLE,4,12340,2086,2022-08-01 00:08:00,2022-12-10 09:26:00
5,RANDOM,5,9618,2001,2022-08-01 03:47:00,2022-12-07 17:36:00
6,BIPARTITE,6,12174,2109,2022-08-01 06:11:00,2022-11-22 07:52:00
7,STACK,7,23624,2096,2022-08-01 11:08:00,2022-12-17 18:04:00
8,UNMATCHED,8,87610,0,2022-08-01 00:01:00,2022-11-05 23:59:00


## 4. 시도 표 → `attempts.parquet`

시각은 **패턴 파일의 시각**(매칭 안 된 행 포함)으로 잡는다. `n_accounts` 는 매칭된 거래의 양끝 계좌 수.

In [7]:
con.execute(f"""
    COPY (
        WITH acct AS (SELECT p.attempt_id, unnest([l.src, l.dst]) AS node FROM pm p JOIN lt l USING (edge_id)),
             na   AS (SELECT attempt_id, count(DISTINCT node) AS n_accounts FROM acct GROUP BY 1),
             own  AS (SELECT attempt_id, count(*) AS n_edges_labeled FROM em GROUP BY 1)
        SELECT p.attempt_id,
               any_value(p.pattern_type)            AS pattern_type,
               any_value(p.pattern_desc)            AS pattern_desc,
               count(*)                             AS n_rows,
               count(DISTINCT p.edge_id)            AS n_edges,
               count(*) FILTER (WHERE p.unmatched)  AS n_rows_unmatched,
               coalesce(any_value(own.n_edges_labeled), 0) AS n_edges_labeled,
               any_value(na.n_accounts)             AS n_accounts,
               min(p.ts) AS start_ts, max(p.ts) AS end_ts,
               date_diff('minute', min(p.ts), max(p.ts)) / 60.0 AS duration_h
        FROM pm p LEFT JOIN na USING (attempt_id) LEFT JOIN own USING (attempt_id)
        GROUP BY p.attempt_id ORDER BY p.attempt_id
    ) TO {q(OUT / 'attempts.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)""")
con.execute(f"COPY (SELECT * FROM daily_all ORDER BY day) TO {q(OUT / 'daily_all.parquet')} (FORMAT PARQUET)")

att = pd.read_parquet(OUT / "attempts.parquet")
att.groupby("pattern_type").agg(attempts=("attempt_id", "size"), rows_med=("n_rows", "median"),
                                accounts_med=("n_accounts", "median"), duration_d_med=("duration_h", lambda s: s.median() / 24),
                                duration_d_max=("duration_h", lambda s: s.max() / 24)).round(2)

,attempts,rows_med,accounts_med,duration_d_med,duration_d_max
pattern_type,,,,,
BIPARTITE,2109,4.0,8.0,10.43,16.95
CYCLE,2086,5.0,5.0,29.18,35.02
FAN-IN,2014,6.0,7.0,28.81,35.02
FAN-OUT,2040,6.0,7.0,29.13,35.01
GATHER-SCATTER,2054,13.0,14.0,55.76,70.45
RANDOM,2001,4.0,5.0,20.52,35.00
SCATTER-GATHER,2067,10.0,7.0,32.44,35.02
STACK,2096,8.0,12.0,26.51,42.51


## 5. 마무리

In [8]:
con.close()
import shutil; shutil.rmtree(tmp, ignore_errors=True)
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
for f in sorted(OUT.glob("*")):
    print(f"{f.name:22s} {f.stat().st_size/1e6:8.2f} MB")

{
  "dataset": "HI-Large",
  "laundering_edges": 225546,
  "last_day_all": "2023-01-12 00:00:00",
  "last_day_normal": "2023-01-12 00:00:00",
  "last_day_laundering": "2023-01-12 00:00:00",
  "pattern_rows": 137936,
  "pattern_rows_unmatched": 0,
  "attempts": 16467,
  "pattern_edges": 137936,
  "edges_multi_attempt": 0,
  "edges_multi_type": 0,
  "edges_dup_in_attempt": 0,
  "pattern_edges_label0": 0,
  "class_edges": {
    "FAN-OUT": 13554,
    "FAN-IN": 13287,
    "GATHER-SCATTER": 26361,
    "SCATTER-GATHER": 26978,
    "CYCLE": 12340,
    "RANDOM": 9618,
    "BIPARTITE": 12174,
    "STACK": 23624,
    "UNMATCHED": 87610
  }
}
attempts.parquet           0.30 MB
daily_all.parquet          0.00 MB
laundering.parquet         5.98 MB
summary.json               0.00 MB
